## How to get a subset of Argo data for a heatwave

first load in our HW table from the ```MHW_list.csv``` file

In [2]:
import pandas as pd
import datetime


# Path to Crocolake
parquet_dir = '/home/jovyan/shared/go-bgc-2026/data/CrocoLake/BGC_CROCOLAKE/'

# path to heatwave csv
csv_path = '../MHW_list.csv'

dfhw = pd.read_csv(csv_path)
dfhw.set_index('name',inplace=True)
date_cols = ['date_start','date_end']
dfhw[date_cols] = dfhw[date_cols].apply(pd.to_datetime)

# Choose a HW
hw = "SWPac2023"

# Boundaries
lat0 = dfhw.loc[hw,'lat0'] 
lat1 = dfhw.loc[hw,'lat1'] 
lon0 = dfhw.loc[hw,'lon0'] 
lon1 = dfhw.loc[hw,'lon1'] 

date0 = dfhw.loc[hw,'date_start']
date1 = dfhw.loc[hw,'date_end']


## CrocoLake Access

## Warning - subset doesn't work yet if crossing 180deg line 

In [3]:
%%time

# Parameters
columns = ('PRES','TEMP','PSAL','DOXY','CHLA','BBP700','LATITUDE','LONGITUDE','JULD','CYCLE_NUMBER','DB_NAME','PLATFORM_NUMBER')


# without DB_NAME you also would get any Spray glider or shipboard GLODAP data too
filters = [
    ("LATITUDE",">",lat0), ("LATITUDE","<",lat1),
    ("LONGITUDE",">",lon0), ("LONGITUDE","<",lon1),
    ("JULD",">",date0), ("JULD","<",date1),
    ("DB_NAME","=","ARGO") 
]

df = pd.read_parquet(parquet_dir,columns=columns,filters=filters)
df = df.dropna(subset=['DOXY','CHLA','BBP700','PSAL','TEMP'])
df

CPU times: user 5.56 s, sys: 706 ms, total: 6.26 s
Wall time: 1.58 s


,PRES,TEMP,PSAL,DOXY,CHLA,BBP700,LATITUDE,LONGITUDE,JULD,CYCLE_NUMBER,DB_NAME,PLATFORM_NUMBER


## example for when lon bounds cross 180 deg line

In [5]:
if lon0 > lon1:
    filters1 = [
    ("LATITUDE",">",lat0), ("LATITUDE","<",lat1),
    ("LONGITUDE",">",lon0), ("LONGITUDE","<=",180),
    ("JULD",">",date0), ("JULD","<",date1),
    ("DB_NAME","=","ARGO") 
    ]
    filters2 = [
    ("LATITUDE",">",lat0), ("LATITUDE","<",lat1),
    ("LONGITUDE",">",-180), ("LONGITUDE","<=",lon1),
    ("JULD",">",date0), ("JULD","<",date1),
    ("DB_NAME","=","ARGO") 
    ]
    df1 = pd.read_parquet(parquet_dir,columns=columns,filters=filters1)
    df2 = pd.read_parquet(parquet_dir,columns=columns,filters=filters2)
    df = pd.concat([df1, df2], ignore_index=True)
df = df.dropna(subset=['DOXY','CHLA','BBP700','PSAL','TEMP'])
df

,PRES,TEMP,PSAL,DOXY,CHLA,BBP700,LATITUDE,LONGITUDE,JULD,CYCLE_NUMBER,DB_NAME,PLATFORM_NUMBER
0,2.16,11.536,34.302765,293.352722,0.703892,0.002301,-49.9702,152.9926,2023-01-12 15:23:42.001642496,1,ARGO,5906443
1,3.96,11.539,34.303764,293.265137,0.701547,0.002425,-49.9702,152.9926,2023-01-12 15:23:42.001642496,1,ARGO,5906443
2,5.96,11.538,34.302765,293.132507,0.710145,0.002349,-49.9702,152.9926,2023-01-12 15:23:42.001642496,1,ARGO,5906443
3,7.96,11.537,34.302765,293.177673,0.700375,0.002385,-49.9702,152.9926,2023-01-12 15:23:42.001642496,1,ARGO,5906443
4,9.96,11.538,34.303764,293.111389,0.708191,0.002435,-49.9702,152.9926,2023-01-12 15:23:42.001642496,1,ARGO,5906443
...,...,...,...,...,...,...,...,...,...,...,...,...
368948,1200.089966,4.4124,34.341183,194.563431,0.002031,0.000201,-40.4363,-167.2653,2023-12-27 19:15:18.002060288,40,ARGO,5906567
368949,1299.910034,3.8301,34.371181,186.182968,0.002031,0.0002,-40.4363,-167.2653,2023-12-27 19:15:18.002060288,40,ARGO,5906567
368950,1399.98999,3.6198,34.41988,176.989899,0.002031,0.000211,-40.4363,-167.2653,2023-12-27 19:15:18.002060288,40,ARGO,5906567
368951,1500.160034,3.3307,34.464283,170.357895,0.004061,0.000222,-40.4363,-167.2653,2023-12-27 19:15:18.002060288,40,ARGO,5906567
